In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

In [ ]:
using_colab = False


In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [ ]:
# Install Florence-2 / BLIP-2 dependencies when needed
import sys
if using_colab:
    !{sys.executable} -m pip install transformers==4.49.0 huggingface_hub<1.0 tokenizers<0.22 einops timm accelerate sentencepiece ipywidgets ipympl
else:
    # Dependencies are expected to be installed from requirements.txt
    pass


In [ ]:
from IPython import get_ipython
ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Using interactive matplotlib widget backend.")
    except Exception as exc:
        ip.run_line_magic("matplotlib", "inline")
        print(f"Widget backend unavailable ({exc}). Falling back to inline output.")
else:
    import matplotlib
    matplotlib.use("Agg")
    print("Running outside IPython; using Agg backend.")


In [ ]:
import torch

# turn on tfloat32 for Ampere GPUs when CUDA is available
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # use bfloat16 for the whole notebook on supported GPUs
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()


In [ ]:
import importlib.machinery
import os
import sys
import tempfile
import types
from pathlib import Path

os.environ["HF_MODULES_CACHE"] = str(Path(tempfile.gettempdir()) / "hf_modules")
Path(os.environ["HF_MODULES_CACHE"]).mkdir(parents=True, exist_ok=True)
HF_CACHE = Path.home() / ".cache" / "huggingface" / "hub"


def install_sam3_compat_shims():
    if "triton" not in sys.modules:
        triton_module = types.ModuleType("triton")
        triton_module.__spec__ = importlib.machinery.ModuleSpec("triton", loader=None)

        def jit(fn=None, **_kwargs):
            if fn is None:
                def decorator(inner):
                    return inner
                return decorator
            return fn

        def autotune(configs=None, key=None):
            def decorator(fn):
                return fn
            return decorator

        def cdiv(x, y):
            return (x + y - 1) // y

        class Config:
            def __init__(self, *args, **kwargs):
                self.args = args
                self.kwargs = kwargs

        triton_module.jit = jit
        triton_module.autotune = autotune
        triton_module.cdiv = cdiv
        triton_module.Config = Config
        sys.modules["triton"] = triton_module

    if "triton.language" not in sys.modules:
        triton_language = types.ModuleType("triton.language")
        triton_language.__spec__ = importlib.machinery.ModuleSpec("triton.language", loader=None)

        class constexpr:
            pass

        triton_language.constexpr = constexpr
        sys.modules["triton.language"] = triton_language
        sys.modules["triton"].language = triton_language

    if "decord" not in sys.modules:
        decord_module = types.ModuleType("decord")
        decord_module.__spec__ = importlib.machinery.ModuleSpec("decord", loader=None)

        def cpu(index=0):
            return index

        class VideoReader:
            def __init__(self, *args, **kwargs):
                raise RuntimeError("decord is not installed in this environment. VideoReader is unavailable.")

        decord_module.cpu = cpu
        decord_module.VideoReader = VideoReader
        sys.modules["decord"] = decord_module

    if "pycocotools" not in sys.modules:
        pycocotools_module = types.ModuleType("pycocotools")
        pycocotools_module.__spec__ = importlib.machinery.ModuleSpec("pycocotools", loader=None)
        pycocotools_module.__path__ = []
        sys.modules["pycocotools"] = pycocotools_module

    if "pycocotools.mask" not in sys.modules:
        pycocotools_mask = types.ModuleType("pycocotools.mask")
        pycocotools_mask.__spec__ = importlib.machinery.ModuleSpec("pycocotools.mask", loader=None)
        sys.modules["pycocotools.mask"] = pycocotools_mask
        sys.modules["pycocotools"].mask = pycocotools_mask


install_sam3_compat_shims()

import sam3
from huggingface_hub import hf_hub_download
from sam3 import build_sam3_image_model

sam3_root = Path(sam3.__file__).resolve().parent
sam_device = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_sam3_checkpoint():
    snapshots_dir = HF_CACHE / "models--facebook--sam3" / "snapshots"
    if snapshots_dir.exists():
        for snapshot in sorted([p for p in snapshots_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True):
            checkpoint = snapshot / "sam3.pt"
            if checkpoint.exists():
                print("Using cached SAM3 checkpoint:", checkpoint)
                return checkpoint
    print("Downloading SAM3 checkpoint from Hugging Face...")
    return Path(hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_files_only=False))


def cached_snapshot(repo_dir_name: str, required_files=None):
    required_files = required_files or []
    snapshots_dir = HF_CACHE / repo_dir_name / "snapshots"
    if snapshots_dir.exists():
        snapshots = sorted([p for p in snapshots_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
        for snapshot in snapshots:
            if all((snapshot / name).exists() for name in required_files):
                return snapshot
    return None


In [ ]:
sam_checkpoint = resolve_sam3_checkpoint()
print(f"Loading SAM3 on {sam_device}...")
model = build_sam3_image_model(
    checkpoint_path=str(sam_checkpoint),
    load_from_HF=False,
    device=sam_device,
)


In [ ]:
from sam3.model.sam3_image_processor import Sam3Processor
processor = Sam3Processor(model, device=sam_device)


In [ ]:
# Florence-2 on a secondary GPU when available
from transformers import AutoProcessor, AutoModelForCausalLM
import os

florence_device = "cuda:1" if torch.cuda.device_count() > 1 else ("cuda:0" if torch.cuda.is_available() else "cpu")
florence_source = cached_snapshot(
    "models--microsoft--Florence-2-large-ft",
    required_files=["config.json", "generation_config.json", "preprocessor_config.json"],
)
if florence_source is None:
    florence_source = "microsoft/Florence-2-large-ft"
print(f"Loading Florence-2-large-ft on {florence_device}...")
try:
    florence_model = AutoModelForCausalLM.from_pretrained(
        florence_source,
        trust_remote_code=True,
        attn_implementation="eager",
        device_map=florence_device,
        local_files_only=isinstance(florence_source, Path),
    ).to(torch.bfloat16 if "cuda" in florence_device else torch.float32)

    florence_processor = AutoProcessor.from_pretrained(
        florence_source,
        trust_remote_code=True,
        local_files_only=isinstance(florence_source, Path),
    )
    print(f"Florence-2-large-ft loaded on {florence_device}.")
except Exception as e:
    print(f"Florence-2 load error: {str(e)}")
    florence_model = None
    florence_processor = None

os.makedirs("cropped_outputs", exist_ok=True)


In [ ]:
# BLIP-2 on a secondary GPU when available
from transformers import Blip2Processor, Blip2ForConditionalGeneration

blip_device = "cuda:1" if torch.cuda.device_count() > 1 else ("cuda:0" if torch.cuda.is_available() else "cpu")
blip_source = cached_snapshot(
    "models--Salesforce--blip2-opt-2.7b",
    required_files=["config.json", "preprocessor_config.json"],
)
if blip_source is None:
    blip_source = "Salesforce/blip2-opt-2.7b"
print(f"Loading BLIP-2 on {blip_device}...")
try:
    blip_processor = Blip2Processor.from_pretrained(
        blip_source,
        local_files_only=isinstance(blip_source, Path),
    )
    blip_model = Blip2ForConditionalGeneration.from_pretrained(
        blip_source,
        torch_dtype=torch.bfloat16 if "cuda" in blip_device else torch.float32,
        device_map=blip_device,
        local_files_only=isinstance(blip_source, Path),
    )
    print(f"BLIP-2 loaded on {blip_device}.")
except Exception as e:
    print(f"BLIP-2 load error: {str(e)}")
    blip_model = None
    blip_processor = None


In [ ]:
import io
import datetime
from pathlib import Path
import textwrap

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import requests
from IPython.display import clear_output, display, HTML
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec
import torch

class Sam3SegmentationWidget:
    """Interactive Jupyter widget for SAM3 segmentation with text and box prompts."""

    def __init__(self, processor, florence_model=None, florence_processor=None, blip_model=None, blip_processor=None):
        self.processor = processor
        self.florence_model = florence_model
        self.florence_processor = florence_processor
        self.blip_model = blip_model
        self.blip_processor = blip_processor

        self.state = None
        self.current_image = None
        self.current_image_array = None
        self.box_mode = "positive"
        self.drawing_box = False
        self.box_start = None
        self.current_rect = None
        
        self.enable_captioning = True

        self._setup_ui()
        self._setup_plot()


    def _setup_ui(self):
        """Set up the UI components with specific Layout."""
        
        # --- 1. Control Widgets ---
        self.upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Upload Image")
        self.upload_widget.observe(self._on_image_upload, names="value")

        self.url_input = widgets.Text(placeholder="Or enter image URL")
        self.url_button = widgets.Button(description="Load URL", button_style="info")
        self.url_button.on_click(self._on_load_url)
        url_box = widgets.HBox([self.url_input, self.url_button], layout=widgets.Layout(width="100%", justify_content="space-between"))

        self.text_input = widgets.Text(placeholder='Click image to detect...', continuous_update=False)
        self.text_input.observe(self._on_text_submit, names="value")
        self.text_button = widgets.Button(description="Segment", button_style="success")
        self.text_button.on_click(self._on_text_prompt)
        text_box = widgets.HBox([self.text_input, self.text_button], layout=widgets.Layout(width="100%", justify_content="space-between"))

        self.box_mode_buttons = widgets.ToggleButtons(
            options=["Positive Boxes", "Negative Boxes"],
            description="Box Mode:",
            tooltips=["Draw/Click to include", "Draw/Click to exclude"]
        )
        self.box_mode_buttons.observe(self._on_box_mode_change, names="value")

        self.clear_button = widgets.Button(description="Clear All Prompts", button_style="warning")
        self.clear_button.on_click(self._on_clear_prompts)

        self.caption_toggle = widgets.Checkbox(value=True, description='Enable Auto-Captioning', style={'description_width': 'initial'})
        self.caption_toggle.observe(self._on_caption_toggle, names='value')

        self.confidence_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="Confidence:", continuous_update=False)
        self.confidence_slider.observe(self._on_confidence_change, names="value")

        self.size_slider = widgets.IntSlider(value=600, min=300, max=2000, step=10, description="Image Size:", continuous_update=False)
        self.size_slider.observe(self._on_size_change, names="value")

        self.status_label = widgets.Label(value="Upload an image to begin")

        # --- 2. Sidebar Layout ---
        source_pane = widgets.VBox([self.upload_widget, url_box])
        prompt_pane = widgets.VBox([widgets.Label("Text Prompt / Detected Label:"), text_box, self.box_mode_buttons, self.confidence_slider, self.caption_toggle, self.clear_button])
        display_pane = widgets.VBox([self.size_slider])

        self.accordion = widgets.Accordion(children=[source_pane, prompt_pane, display_pane])
        self.accordion.set_title(0, "Image Source")
        self.accordion.set_title(1, "Segmentation Prompts")
        self.accordion.set_title(2, "Display Settings")
        self.accordion.selected_index = 0

        sidebar = widgets.VBox([self.status_label, widgets.HTML("<h4>Controls</h4>"), self.accordion])
        sidebar.layout = widgets.Layout(width="380px", min_width="380px", border="1px solid #e0e0e0", padding="10px", margin="0 15px 0 0", flex="0 0 auto")

        # --- 3. Output Widgets ---
        
        self.main_output = widgets.Output() 
        self.plot_container = widgets.Box([self.main_output])
        self.plot_container.add_class("no-drag")
        
        self.comparison_output = widgets.Output()
        
        # --- 4. Final Layout Construction ---
        
        main_area = widgets.VBox([self.plot_container])
        main_area.layout = widgets.Layout(flex="1", min_width="500px", overflow="auto")
        
        top_row = widgets.HBox([sidebar, main_area])
        top_row.layout = widgets.Layout(width="100%", display="flex", flex_flow="row", align_items="stretch")

        css_style = widgets.HTML("""<style>.jupyter-matplotlib-canvas, canvas {cursor: crosshair !important;}</style>""")
        
        self.container = widgets.VBox([
            css_style,
            widgets.HTML("<h3>🖼️ SAM3 Interactive Segmentation</h3>"),
            top_row,                  
            widgets.HTML("<hr>"),     
            self.comparison_output    
        ])

    def _setup_plot(self):
        """Set up the matplotlib figure."""
        was_interactive = plt.isinteractive()
        plt.ioff()
        try:
            self.fig, self.ax = plt.subplots(figsize=(8, 6))
            self.ax.axis("off")
            self.fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
            self.fig.canvas.toolbar_visible = False
            self.fig.canvas.header_visible = False
            self.fig.canvas.footer_visible = False
            self.fig.canvas.resizable = False
        finally:
            if was_interactive:
                plt.ion()

    def _set_loading(self, is_loading, message="Processing..."):
        if is_loading:
            self.status_label.value = f"⏳ {message}"
            self.upload_widget.disabled = True
            self.text_button.disabled = True
        else:
            self.upload_widget.disabled = False
            self.text_button.disabled = False

    def _on_image_upload(self, change):
        if change["new"]:
            uploaded_file = change["new"][0]
            image = PIL.Image.open(io.BytesIO(uploaded_file["content"])).convert("RGB")
            self._set_image(image)

    def _on_load_url(self, button):
        url = self.url_input.value.strip()
        if not url: return
        self._set_loading(True, "Downloading...")
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            image = PIL.Image.open(io.BytesIO(response.content)).convert("RGB")
            self._set_image(image)
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error: {str(e)}"

    def _set_image(self, image):
        self._set_loading(True, "Processing image...")
        with self.comparison_output:
            clear_output()

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()  # Clear GPU memory before loading new image state
            self.current_image = image
            self.current_image_array = np.array(image)
            self.state = self.processor.set_image(image)
            self._set_loading(False)
            self.status_label.value = f"Image loaded: {image.size}"
            self._resize_figure()
            self._update_display() 
            self._connect_plot_events()
            self.accordion.selected_index = 1
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error: {str(e)}"

    def _on_text_submit(self, change):
        self._on_text_prompt(None)

    def _on_text_prompt(self, button):
        """Standard Text Prompt Handler."""
        if self.state is None: return
        prompt = self.text_input.value.strip()
        if not prompt: return

        self._set_loading(True, f"Segmenting '{prompt}'...")
        try:
            torch.cuda.empty_cache()
            self.state = self.processor.set_text_prompt(prompt, self.state)
            self._process_and_display_results(prompt) 
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error: {str(e)}"

    def _on_box_mode_change(self, change):
        self.box_mode = "positive" if change["new"] == "Positive Boxes" else "negative"

    def _on_clear_prompts(self, button):
        if self.current_image is not None:
            self.state = self.processor.reset_all_prompts(self.state)
            if "prompted_boxes" in self.state: del self.state["prompted_boxes"]
            self.text_input.value = ""
            self._update_display()
            with self.comparison_output:
                clear_output()
            self.status_label.value = "Cleared prompts"

    def _on_confidence_change(self, change):
        if self.state is not None:
            self.state = self.processor.set_confidence_threshold(change["new"], self.state)
            self._update_display()

    # --- Plotting Events ---
    def _connect_plot_events(self):
        if hasattr(self.fig.canvas, "toolbar") and self.fig.canvas.toolbar:
            self.fig.canvas.toolbar.pan(); self.fig.canvas.toolbar.pan()
        self.fig.canvas.mpl_connect("button_press_event", self._on_press)
        self.fig.canvas.mpl_connect("button_release_event", self._on_release)
        self.fig.canvas.mpl_connect("motion_notify_event", self._on_motion)

    def _on_press(self, event):
        if event.inaxes != self.ax: return
        self.drawing_box = True
        self.box_start = (event.xdata, event.ydata)

    def _on_motion(self, event):
        if not self.drawing_box or event.inaxes != self.ax or not self.box_start: return
        if self.current_rect: self.current_rect.remove()
        x0, y0 = self.box_start
        self.current_rect = Rectangle((x0, y0), event.xdata-x0, event.ydata-y0, fill=False, edgecolor="green" if self.box_mode=="positive" else "red", linewidth=2, linestyle="--")
        self.ax.add_patch(self.current_rect)
        self.fig.canvas.draw_idle()

    def _on_release(self, event):
        """Handle Mouse Release: Distinguish between Click (Point) and Drag (Box)."""
        if not self.drawing_box or event.inaxes != self.ax or not self.box_start:
            self.drawing_box = False; return
        self.drawing_box = False
        if self.current_rect: self.current_rect.remove(); self.current_rect = None
        if self.state is None: return

        x0, y0 = self.box_start
        x1, y1 = event.xdata, event.ydata
        
        # --- COORDINATE NORMALIZATION ---
        img_h, img_w = self.state["original_height"], self.state["original_width"]
        
        # Check if it's a CLICK (Point) or a DRAG (Box)
        is_click = abs(x1-x0) < 5 and abs(y1-y0) < 5
        
        if is_click:
            # === HANDLE CLICK (POINT PROMPT) ===
            self._set_loading(True, "Auto-detecting object at click...")
            
            # FIXED: Create a larger 10x10px box to prevent CUBLAS/Math errors
            half_size = 5 # 5 pixels on each side = 10px box
            x_min, x_max = max(0, x0 - half_size), min(img_w, x0 + half_size)
            y_min, y_max = max(0, y0 - half_size), min(img_h, y0 + half_size)
            
            # Convert to normalized cx, cy, w, h
            width = (x_max - x_min) / img_w
            height = (y_max - y_min) / img_h
            center_x = (x_min + x_max) / 2.0 / img_w
            center_y = (y_min + y_max) / 2.0 / img_h
            
            box = [center_x, center_y, width, height]
            label = self.box_mode == "positive"
            
            # Store for display
            if "prompted_boxes" not in self.state: self.state["prompted_boxes"] = []
            self.state["prompted_boxes"].append({"box": [x_min, y_min, x_max, y_max], "label": label})
            
            try:
                # 1. Clear GPU cache to prevent CUBLAS ERROR
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
                # 2. Segment
                self.state = self.processor.add_geometric_prompt(box, label, self.state)
                
                # 3. AUTO-LABEL LOGIC
                if self.state and "boxes" in self.state and len(self.state["boxes"]) > 0:
                    # Find Largest Box (Smart Selection)
                    boxes = self.state["boxes"].cpu().numpy()
                    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                    largest_idx = np.argmax(areas)
                    target_bbox = boxes[largest_idx]
                    
                    # Crop and Identify
                    cropped_pil, _ = self._crop_and_save_image(target_bbox)
                    
                    if cropped_pil:
                        # Use Florence-2 to Identify the object (Short caption)
                        auto_label = self._generate_caption(cropped_pil, task_prompt="<CAPTION>")
                        self.text_input.value = auto_label
                        
                        # Trigger display
                        self._process_and_display_results(auto_label)
                    else:
                        self._update_display()
                        self._set_loading(False)
                else:
                    self._update_display()
                    self._set_loading(False)
                    self.status_label.value = "No object detected at click location"
                    
            except Exception as e:
                self._set_loading(False)
                self.status_label.value = f"Error: {str(e)}"
                import traceback
                traceback.print_exc()
                
        else:
            # === HANDLE DRAG (EXISTING BOX LOGIC) ===
            x_min, x_max = min(x0,x1), max(x0,x1)
            y_min, y_max = min(y0,y1), max(y0,y1)
            
            width = (x_max - x_min) / img_w
            height = (y_max - y_min) / img_h
            center_x = (x_min + x_max) / 2.0 / img_w
            center_y = (y_min + y_max) / 2.0 / img_h
            
            box = [center_x, center_y, width, height]
            label = self.box_mode == "positive"
            
            if "prompted_boxes" not in self.state: self.state["prompted_boxes"] = []
            self.state["prompted_boxes"].append({"box": [x_min, y_min, x_max, y_max], "label": label})

            self._set_loading(True, "Adding box...")
            try:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()  # Clear cache
                self.state = self.processor.add_geometric_prompt(box, label, self.state)
                self._update_display()
                self._set_loading(False)
            except Exception as e:
                self._set_loading(False); self.status_label.value = str(e)

    def _process_and_display_results(self, prompt_text):
        """Helper to update main display and generate captions."""
        self._update_display()
        self.status_label.value = f"Detected: '{prompt_text}'. Generating analysis..."

        if self.enable_captioning and self.florence_model is not None:
            if self.state is not None and "boxes" in self.state and len(self.state["boxes"]) > 0:
                
                # Find largest box
                boxes = self.state["boxes"].cpu().numpy()
                areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                largest_idx = np.argmax(areas)
                target_bbox = boxes[largest_idx]
                
                # Crop
                cropped_pil, saved_filename = self._crop_and_save_image(target_bbox)

                if cropped_pil is not None:
                    original_pil = PIL.Image.fromarray(self.current_image_array)
                    
                    # Generate DETAILED Captions for comparison
                    original_caption = self._generate_caption(original_pil, task_prompt="<DETAILED_CAPTION>")
                    cropped_caption = self._generate_caption(cropped_pil, task_prompt="<DETAILED_CAPTION>")
                    
                    self._display_comparison(original_pil, cropped_pil, original_caption, cropped_caption)
                    self.status_label.value = f"✅ Auto-labeled: '{prompt_text}'"
        
        self._set_loading(False)

    def _resize_figure(self):
        if self.current_image is None: return
        img_w, img_h = self.current_image.size
        display_w = float(self.size_slider.value)
        self.fig.set_size_inches((display_w/self.fig.dpi, int(display_w*(img_h/img_w))/self.fig.dpi), forward=True)

    def _on_size_change(self, change):
        if self.current_image:
            self._resize_figure()
            self._update_display()

    # --- MAIN IMAGE DISPLAY UPDATE ---
    def _update_display(self):
        if self.current_image_array is None: return
        
        with self.main_output:
            clear_output(wait=True)
            self.ax.clear(); self.ax.axis("off")
            self.ax.imshow(self.current_image_array)

            if self.state and "masks" in self.state and len(self.state.get("masks", [])) > 0:
                masks = self.state["masks"]
                boxes = self.state["boxes"]
                scores = self.state["scores"]
                mask_overlay = np.zeros((*self.current_image_array.shape[:2], 4))
                
                for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
                    color = plt.cm.tab10(i % 10)[:3]
                    mask_overlay[mask[0].cpu().numpy() > 0.5] = (*color, 0.5)
                    x0, y0, x1, y1 = box.cpu().numpy()
                    self.ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor=color, linewidth=2))
                    self.ax.text(x0, y0-5, f"{score:.2f}", color="white", fontsize=10, bbox=dict(facecolor=color, alpha=0.7, edgecolor="none", pad=2))
                self.ax.imshow(mask_overlay)

            if self.state and "prompted_boxes" in self.state:
                for pb in self.state["prompted_boxes"]:
                    x0, y0, x1, y1 = pb["box"]
                    self.ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="green" if pb["label"] else "red", linewidth=2, linestyle="--"))

            display(self.fig.canvas)

    def _on_caption_toggle(self, change):
        self.enable_captioning = change['new']

    # --- GENERATION & COMPARISON LOGIC ---
    
    def _generate_caption(self, image_pil, task_prompt="<DETAILED_CAPTION>"):
        if not self.florence_model: return "Model not loaded"
        try:
            device = next(self.florence_model.parameters()).device
            inputs = self.florence_processor(text=task_prompt, images=image_pil, return_tensors="pt")
            dtype = torch.bfloat16 if "cuda" in str(device) else torch.float32
            inputs = {k: (v.to(device=device, dtype=dtype) if torch.is_floating_point(v) else v.to(device)) for k, v in inputs.items()}
            generated_ids = self.florence_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3, do_sample=False)
            generated_text = self.florence_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
            parsed = self.florence_processor.post_process_generation(generated_text, task=task_prompt, image_size=(image_pil.width, image_pil.height))
            return parsed.get(task_prompt, "No caption")
        except Exception as e: return f"Error: {str(e)}"

    def _generate_blip_caption(self, image_pil):
        if not self.blip_model: return "Model not loaded"
        try:
            device = next(self.blip_model.parameters()).device
            text_prompt = "Question: Describe this image in detail. Answer:"
            inputs = self.blip_processor(images=image_pil, text=text_prompt, return_tensors="pt")
            dtype = torch.bfloat16 if "cuda" in str(device) else torch.float32
            inputs = {k: (v.to(device=device, dtype=dtype) if torch.is_floating_point(v) else v.to(device)) for k, v in inputs.items()}
            generated_ids = self.blip_model.generate(**inputs, max_new_tokens=250, min_length=30, num_beams=5)
            return self.blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        except Exception as e: return f"Error: {str(e)}"

    def _crop_and_save_image(self, bbox):
        if self.current_image_array is None: return None, None
        try:
            x1, y1, x2, y2 = map(int, bbox)
            x1 = max(0, x1); y1 = max(0, y1)
            x2 = min(self.current_image_array.shape[1], x2); y2 = min(self.current_image_array.shape[0], y2)
            
            cropped_pil = PIL.Image.fromarray(self.current_image_array[y1:y2, x1:x2])
            filename = f"cropped_outputs/crop_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.png"
            cropped_pil.save(filename)
            return cropped_pil, filename
        except Exception: return None, None

    def _display_comparison(self, original_pil, cropped_pil, original_caption, cropped_caption):
        with self.comparison_output:
            clear_output(wait=True) 
            blip_original = self._generate_blip_caption(original_pil)
            blip_cropped = self._generate_blip_caption(cropped_pil)
            
            was_interactive = plt.isinteractive()
            plt.ioff()
            try:
                fig = plt.figure(figsize=(18, 16))
                gs = GridSpec(4, 2, figure=fig, height_ratios=[3, 1, 3, 1], hspace=0.4, wspace=0.2)
                
                # Florence-2 Section
                ax1 = fig.add_subplot(gs[0, 0]); ax1.imshow(np.array(original_pil)); ax1.set_title("Original (Florence-2)", fontsize=14, fontweight='bold'); ax1.axis('off')
                ax2 = fig.add_subplot(gs[0, 1]); ax2.imshow(np.array(cropped_pil)); ax2.set_title("Cropped (Florence-2)", fontsize=14, fontweight='bold'); ax2.axis('off')
                
                ax3 = fig.add_subplot(gs[1, 0]); ax3.axis('off')
                ax3.text(0.5, 0.5, "\n".join(textwrap.wrap(original_caption, 60)), ha='center', va='center', fontsize=11, bbox=dict(boxstyle='round,pad=1', facecolor='wheat', alpha=0.9))
                
                ax4 = fig.add_subplot(gs[1, 1]); ax4.axis('off')
                ax4.text(0.5, 0.5, "\n".join(textwrap.wrap(cropped_caption, 60)), ha='center', va='center', fontsize=11, bbox=dict(boxstyle='round,pad=1', facecolor='lightblue', alpha=0.9))
                
                # BLIP-2 Section
                ax5 = fig.add_subplot(gs[2, 0]); ax5.imshow(np.array(original_pil)); ax5.set_title("Original (BLIP-2)", fontsize=14, fontweight='bold'); ax5.axis('off')
                ax6 = fig.add_subplot(gs[2, 1]); ax6.imshow(np.array(cropped_pil)); ax6.set_title("Cropped (BLIP-2)", fontsize=14, fontweight='bold'); ax6.axis('off')
                
                ax7 = fig.add_subplot(gs[3, 0]); ax7.axis('off')
                ax7.text(0.5, 0.5, "\n".join(textwrap.wrap(blip_original, 60)), ha='center', va='center', fontsize=11, bbox=dict(boxstyle='round,pad=1', facecolor='lightgreen', alpha=0.9))
                
                ax8 = fig.add_subplot(gs[3, 1]); ax8.axis('off')
                ax8.text(0.5, 0.5, "\n".join(textwrap.wrap(blip_cropped, 60)), ha='center', va='center', fontsize=11, bbox=dict(boxstyle='round,pad=1', facecolor='lightcoral', alpha=0.9))
                
                fig.suptitle('SAM3 Segmentation + VLM Analysis', fontsize=16, fontweight='bold', y=0.95)
            finally:
                if was_interactive: plt.ion()

            display(fig)
            plt.close(fig)

    def display(self):
        display(self.container)

    def _ipython_display_(self):
        self.display()



In [ ]:
widget = Sam3SegmentationWidget(
    processor, 
    florence_model=florence_model, 
    florence_processor=florence_processor,
    blip_model=blip_model,
    blip_processor=blip_processor
)
widget.display()